In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-101", revision="no_timm")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-101", revision="no_timm")

model.to(device)

input_folder = "/home/ajeet/codework/datasets/phone_classifier_dataset/person_holding_a_phone/merge_all"
output_folder = "cropped_images/person_holding_a_phone"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    if img_name.lower().endswith(('.jpg')):
        print(f"Processing image: {img_name}")

        image = Image.open(img_path)
        image = image.resize((320, 240))

        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]], device=device)
        results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.3)[0]

        for idx, (score, label, box) in enumerate(zip(results["scores"], results["labels"], results["boxes"])):
            box = [round(i, 2) for i in box.tolist()]
            if label.item() == 77:
                print(f"Detected {model.config.id2label[label.item()]} with confidence {round(score.item(), 3)} at location {box}")
                cropped_image = image.crop((box[0], box[1], box[2], box[3]))
                to_save = img_name.split(".")[0]
                cropped_image_path = os.path.join(output_folder, f"{to_save}_cropped.jpg")
                cropped_image.save(cropped_image_path)
                print(f"Cropped image saved at: {cropped_image_path}")

    # break


In [ ]:
import os
import numpy as np
from PIL import Image, ImageDraw, ImageFont

def create_sprite_image(images, filenames, grid_size=(10, 10), sprite_size=64):
    n_images = len(images)
    actual_grid_size = (min(grid_size[0], n_images), (n_images + grid_size[0] - 1) // grid_size[0])
    sprite_image = Image.new('RGB', (sprite_size * actual_grid_size[0], sprite_size * actual_grid_size[1]))

    for index, (img, filename) in enumerate(zip(images, filenames)):
        img = img.resize((sprite_size, sprite_size))
        draw = ImageDraw.Draw(img)
        label = os.path.basename(filename).split('.')[1]
        draw.text((5, 5), label, fill="green")
        x = (index % grid_size[0]) * sprite_size
        y = (index // grid_size[0]) * sprite_size
        sprite_image.paste(img, (x, y))

    return sprite_image


def create_sprites_from_folder(image_folder, grid_size=(10, 10), sprite_size=64):
    image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png'))]
    if len(image_paths) == 0:
        raise ValueError(f"No images found in the folder {image_folder}.")

    chunk_size = grid_size[0] * grid_size[1]
    for i in range(0, len(image_paths), chunk_size):
        chunk_paths = image_paths[i:i + chunk_size]
        images = [Image.open(path) for path in chunk_paths]
        filenames = [os.path.basename(path) for path in chunk_paths]
        sprite_image = create_sprite_image(images, filenames, grid_size, sprite_size)
        sprite_filename = f"sprite_{i // chunk_size + 1}.png"
        sprite_path = os.path.join("/home/ajeet/codework/visionworkajeet/models/my_visionwork/models/clip/cropped_images/sprit_internet", sprite_filename)
        sprite_image.save(sprite_path)
        print(f"Saved sprite image: {sprite_path}")

image_folder = "../cropped_images/person_holding_a_phone"
create_sprites_from_folder(image_folder)
